# TP 5LAKE — Partie 5 : Analytics SQL & Business Intelligence

**Source d'analyse :** `workspace.5lake_lab.silver_trips_pipeline` (sortie Silver du pipeline Lakeflow).  
**Objectif :** analyser les courses NYC Taxi et documenter les graphiques du dashboard AI/BI et les contrôles Genie.

> **Important :** `tip_pct` est une valeur **synthétique**, et non un pourboire réellement observé. Les résultats de ce notebook sont à recalculer après chaque mise à jour du pipeline. Les sorties du notebook historique `silver_trips` ne constituent pas une vérification de `silver_trips_pipeline`.

Chaque requête est en `%sql` pour fonctionner dans un notebook Databricks dont le langage par défaut est Python.

## 0 — Vérifier la source

Confirmer les droits `USE CATALOG`, `USE SCHEMA` et `SELECT` ainsi que la présence des colonnes nécessaires. La requête ci-dessous affiche le volume réel de courses disponibles dans le pipeline.

In [ ]:
%sql
SELECT COUNT(*) AS total_courses
FROM workspace.`5lake_lab`.silver_trips_pipeline;

## 1 — Top 10 des journées les plus chargées

Regroupement par date de prise en charge et classement décroissant. **Graphique :** barres (date / nombre de courses).

In [ ]:
%sql
SELECT
    DATE(tpep_pickup_datetime) AS jour,
    COUNT(*) AS nombre_courses
FROM workspace.`5lake_lab`.silver_trips_pipeline
GROUP BY DATE(tpep_pickup_datetime)
ORDER BY nombre_courses DESC
LIMIT 10;

## 2 — Distance parcourue et pourboire moyen

On regroupe les distances en tranches d'un mile, on calcule `AVG(tip_pct)`, et on écarte les groupes de moins de 20 courses pour la lisibilité. **Graphique :** courbe, `distance_miles` en X, `pourboire_moyen_pct` en Y. Un résultat agrégé sur toute la période ne suit pas automatiquement un filtre journalier du dashboard.

In [ ]:
%sql
SELECT
    FLOOR(trip_distance) AS distance_miles,
    ROUND(AVG(tip_pct), 2) AS pourboire_moyen_pct,
    COUNT(*) AS nombre_courses
FROM workspace.`5lake_lab`.silver_trips_pipeline
WHERE trip_distance > 0
  AND trip_distance <= 30
  AND tip_pct IS NOT NULL
GROUP BY FLOOR(trip_distance)
HAVING COUNT(*) >= 20
ORDER BY distance_miles;

## 3 — Répartition des courses par heure

Extraction de l'heure de départ et comptage par tranche horaire. **Graphique :** barres de 0 à 23 heures.

In [ ]:
%sql
SELECT
    HOUR(tpep_pickup_datetime) AS heure,
    COUNT(*) AS nombre_courses
FROM workspace.`5lake_lab`.silver_trips_pipeline
GROUP BY HOUR(tpep_pickup_datetime)
ORDER BY heure;

## 4 — Comparaison semaine / week-end

Sur Databricks SQL, `DAYOFWEEK()` renvoie 1 pour dimanche et 7 pour samedi. On compare volume, distance moyenne et pourcentage moyen de pourboire. **Graphique :** barres (`periode`, `pourboire_moyen_pct`).

In [ ]:
%sql
SELECT
    CASE
        WHEN DAYOFWEEK(tpep_pickup_datetime) IN (1, 7)
            THEN 'Week-end'
        ELSE 'Semaine'
    END AS periode,
    COUNT(*) AS nombre_courses,
    ROUND(AVG(trip_distance), 2) AS distance_moyenne,
    ROUND(AVG(tip_pct), 2) AS pourboire_moyen_pct
FROM workspace.`5lake_lab`.silver_trips_pipeline
GROUP BY 1
ORDER BY periode;

## 5 — Dataset pour le dashboard AI/BI

Le dataset ci-dessous expose les données au niveau de chaque course, afin que les trois visualisations puissent être reliées à un **même filtre de date**. À copier dans **Dashboards → Data → Add SQL dataset**, puis enregistrer sous `courses_nyc` (ne pas exécuter le code suivant pour créer une table).

- **Courses par jour** : X = `jour`, Y = `COUNT(*)`
- **Courses par heure** : X = `heure`, Y = `COUNT(*)`
- **Pourboire selon distance** : X = `distance_miles`, Y = `AVG(tip_pct)`
- **Filtre global** : `jour` (plage de dates si disponible)

Lorsque le pipeline est réactualisé, rafraîchir les datasets du dashboard et republier.

In [ ]:
%sql
SELECT
    DATE(tpep_pickup_datetime) AS jour,
    HOUR(tpep_pickup_datetime) AS heure,
    FLOOR(trip_distance) AS distance_miles,
    trip_distance,
    fare_amount,
    tip_pct,
    CASE
        WHEN DAYOFWEEK(tpep_pickup_datetime) IN (1, 7)
            THEN 'Week-end'
        ELSE 'Semaine'
    END AS periode
FROM workspace.`5lake_lab`.silver_trips_pipeline
WHERE trip_distance > 0
  AND fare_amount > 0;

## 6 — Contrôle des réponses Genie

Agent : **Taxi Trip Performance Analytics**. Ajouter `workspace.5lake_lab.silver_trips_pipeline` dans **Configure → Sources**, puis poser les questions suivantes :

1. « Combien de courses contient la table `silver_trips_pipeline` ? »
2. « Quels sont les dix jours avec le plus grand nombre de courses dans `silver_trips_pipeline` ? »
3. « Quel est le pourcentage moyen de pourboire en semaine et le week-end dans `silver_trips_pipeline` ? »

Comparer le SQL généré et les résultats de Genie avec les requêtes de ce notebook, **sur la même table et le même instantané**. Un écart entre `silver_trips` et `silver_trips_pipeline` ne prouve pas à lui seul une erreur : ces tables peuvent utiliser des pourboires synthétiques différents.

**Problème observé :** Genie ne reconnaissait pas initialement `silver_trips_pipeline`, faute de source configurée. Il a fallu l'ajouter dans **Configure → Sources**.

**Note de rendu :** enregistrer une capture des réponses et, idéalement, du SQL généré.

## 7 — Conclusion

Cette partie exploite les données Silver produites par Lakeflow pour analyser la volumétrie et les pourboires synthétiques des courses. Le dashboard AI/BI facilite l'exploration visuelle et Genie permet d'interroger les données en langage naturel, sous réserve de contrôler ses requêtes. Le Machine Learning, MLflow et le scoring dans `gold_predictions` sont documentés dans les notebooks distincts du projet.